In [0]:
from pyspark.sql import functions as F

acs = spark.read.csv("/Volumes/workspace/default/health_data_cs483/ACS_2016_2020.csv",
                     header=True, inferSchema=True)

acs_clean = (acs
    .drop("OBJECTID_1", "Shape__Area", "Shape__Length")   # map/ID columns, not needed
    .filter(F.col("NAME") != "Puerto Rico")                # other datasets are states + DC
    .withColumnRenamed("GEOID", "statefips")               # join key for later
    .withColumnRenamed("STUSAB", "state_abbr")
    .withColumnRenamed("NAME", "state_name")
    .withColumnRenamed("B19013EST1", "median_household_income")
    .withColumnRenamed("B17021EST2_PCT", "poverty_pct")
    .withColumnRenamed("B23001_UE_PCT", "unemployment_pct"))

print(acs.count(), "→", acs_clean.count())
acs_clean.write.mode("overwrite").saveAsTable("workspace.default.acs_clean")

display(spark.table("workspace.default.acs_clean"))